In [3]:
import os
import shutil

OLD_DATASET = "../data/speech_commands_v0.01"
OUT_DIR = "../Dataset"

TEST_LIST = "../data/speech_commands_v0.01/testing_list.txt"
VAL_LIST = "../data/speech_commands_v0.01/validation_list.txt"

def make_dirs():
    for subset in ["train", "val", "test"]:
        for cls in ["0", "1"]:
            os.makedirs(os.path.join(OUT_DIR, subset, cls), exist_ok=True)

def read_list(file_path):
    with open(file_path, "r") as f:
        return set([line.strip() for line in f if line.strip()])

def copy_files(file_set, subset):
    for rel_path in file_set:
        folder, filename = rel_path.split("/")

        src = os.path.join(OLD_DATASET, folder, filename)

        cls = "1" if folder == "stop" else "0"

        dst = os.path.join(OUT_DIR, subset, cls, filename)
        shutil.copy2(src, dst)

def main():
    make_dirs()

    test_set = read_list(TEST_LIST)
    val_set = read_list(VAL_LIST)

    # Copy test and validation
    copy_files(test_set, "test")
    copy_files(val_set, "val")

    # Create train set = everything else
    all_files = []
    for folder in os.listdir(OLD_DATASET):
        if folder == "_background_noise_":
            continue
        folder_path = os.path.join(OLD_DATASET, folder)
        for f in os.listdir(folder_path):
            if f.endswith(".wav"):
                all_files.append(f"{folder}/{f}")

    train_set = [f for f in all_files if f not in test_set and f not in val_set]
    copy_files(train_set, "train")

    print("Done! Train/val/test created in Dataset/")

if __name__ == "__main__":
    main()


Done! Train/val/test created in Dataset/


In [1]:
import os
import random
import torch
import torchaudio
import torch.nn as nn
import torch.nn.functional as F
import soundfile as sf
from torch.utils.data import Dataset, DataLoader

CONFIG – global settings

What it does:
Defines all core constants for training and audio processing.

Why it exists:
So you don’t hardcode numbers everywhere and regret your life later.

Includes:
- sample rate (16kHz)
- clip length (1 sec)
- batch size, epochs
- early stopping patience
- device (CPU/GPU)
- dataset folder paths

If something needs tweaking, it’s usually here first.

In [2]:
SAMPLE_RATE = 16000
CLIP_LEN = SAMPLE_RATE
BATCH_SIZE = 32
EPOCHS = 40
PATIENCE = 5
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

BASE = "../Dataset"
TRAIN_DIR = os.path.join(BASE, "train")
VAL_DIR   = os.path.join(BASE, "val")
TEST_DIR  = os.path.join(BASE, "test")

SAFE AUDIO LOAD – reliable wav reader

What it does:
Loads .wav files safely using soundfile instead of torchaudio’s cursed decoder stack.

Steps:

- read wav
- convert to tensor
- stereo → mono
- resample to 16kHz
- return [1, samples] tensor

Why it exists:
Windows + torchaudio + torchcodec = chaos.
This keeps loading stable and predictable.

Every audio file passes through this function.

In [3]:
def load_wav(path):
    wav, sr = sf.read(path)
    wav = torch.tensor(wav).float()

    # stereo → mono
    if len(wav.shape) > 1:
        wav = wav.mean(dim=1)

    wav = wav.unsqueeze(0)  # [1, samples]

    if sr != SAMPLE_RATE:
        wav = torchaudio.functional.resample(wav, sr, SAMPLE_RATE)

    return wav

DATASET – data pipeline

What it does:
Builds the PyTorch dataset object used by DataLoader.

Responsibilities:
- scans folder structure (0 vs 1)
- loads audio, pads/trims to 1 sec
- applies augmentation (train only)
- converts waveform → log-mel spectrogram
- returns (mel, label)

Why it exists:
This is the entire preprocessing pipeline in one place.

If the model behaves weird, the bug is usually here.

In [4]:
class StopDataset(Dataset):
    def __init__(self, root, augment=False):
        self.files = []
        self.labels = []
        self.augment = augment

        for label in ["0", "1"]:
            folder = os.path.join(root, label)
            for f in os.listdir(folder):
                if f.endswith(".wav"):
                    self.files.append(os.path.join(folder, f))
                    self.labels.append(int(label))

        self.mel = torchaudio.transforms.MelSpectrogram(
            sample_rate=SAMPLE_RATE,
            n_mels=40,
            n_fft=400,
            hop_length=160
        )
        self.db = torchaudio.transforms.AmplitudeToDB()

    def __len__(self):
        return len(self.files)

    def add_noise(self, wav):
        return wav + torch.randn_like(wav) * 0.005

    def time_shift(self, wav):
        shift = int(random.uniform(-0.1, 0.1) * SAMPLE_RATE)
        return torch.roll(wav, shifts=shift, dims=1)

    def gain(self, wav):
        return wav * random.uniform(0.7, 1.3)

    def __getitem__(self, idx):
        wav = load_wav(self.files[idx])

        if wav.shape[1] < CLIP_LEN:
            wav = F.pad(wav, (0, CLIP_LEN - wav.shape[1]))
        else:
            wav = wav[:, :CLIP_LEN]

        if self.augment:
            if random.random() < 0.5:
                wav = self.time_shift(wav)
            if random.random() < 0.5:
                wav = self.gain(wav)
            if random.random() < 0.5:
                wav = self.add_noise(wav)

        mel = self.mel(wav)
        mel = self.db(mel)

        return mel, torch.tensor(self.labels[idx]).float()

MODEL – neural network

What it does:
Defines the tiny CNN that detects the keyword.

Architecture: 3 convolution blocks, pooling, global average pooling, single output neuron

Why it exists:
- Small
- fast
- deployable on Raspberry Pi.

No transformer nonsense. No overengineering.

Outputs logits, not probabilities.

In [5]:
class TinyKWS(nn.Module):
    def __init__(self):
        super().__init__()
        self.conv1 = nn.Conv2d(1, 16, 3, padding=1)
        self.conv2 = nn.Conv2d(16, 32, 3, padding=1)
        self.conv3 = nn.Conv2d(32, 64, 3, padding=1)
        self.pool = nn.MaxPool2d(2)
        self.gap = nn.AdaptiveAvgPool2d(1)
        self.fc = nn.Linear(64, 1)

    def forward(self, x):
        x = F.relu(self.conv1(x))
        x = self.pool(x)

        x = F.relu(self.conv2(x))
        x = self.pool(x)

        x = F.relu(self.conv3(x))
        x = self.gap(x).squeeze(-1).squeeze(-1)
        x = self.fc(x)
        return x

LOAD DATA – dataloaders

What it does:
Creates:

- train dataset
- validation dataset
- test dataset

Wraps them in DataLoader for batching.

Also computes class imbalance weight so the model doesn’t ignore the rare class (“stop”).

Why it exists:
Separates training vs validation behavior and ensures balanced learning.

In [6]:
train_ds = StopDataset(TRAIN_DIR, augment=True)
val_ds   = StopDataset(VAL_DIR, augment=False)
test_ds  = StopDataset(TEST_DIR, augment=False)

train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True)
val_loader   = DataLoader(val_ds, batch_size=BATCH_SIZE)
test_loader  = DataLoader(test_ds, batch_size=BATCH_SIZE)

# imbalance weight
pos = sum(train_ds.labels)
neg = len(train_ds.labels) - pos
pos_weight = torch.tensor([neg / (pos + 1e-6)]).to(DEVICE)

SETUP – training components

What it does:
Initializes everything needed for training.

- model
- optimizer (Adam)
- loss function (BCEWithLogits)
- LR scheduler
- early stopping variables

Why it exists:
This is the training control center.
All knobs live here.

In [7]:
model = TinyKWS().to(DEVICE)
optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)
loss_fn = nn.BCEWithLogitsLoss(pos_weight=pos_weight)

scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
    optimizer, patience=2, factor=0.5
)

best_val = 1e9
counter = 0

TRAIN – main training loop

What it does:
Runs epochs and updates model weights.

Per epoch:

- train on batches
- compute validation loss
- adjust LR if plateau
- save best model
- early stop if no improvement

Why it exists:
Teaches the model without letting it overfit itself into uselessness.

Saves best_model.pt automatically.

In [8]:
for epoch in range(EPOCHS):
    model.train()
    train_loss = 0

    for x, y in train_loader:
        x, y = x.to(DEVICE), y.to(DEVICE).unsqueeze(1)

        pred = model(x)
        loss = loss_fn(pred, y)

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        train_loss += loss.item()

    model.eval()
    val_loss = 0

    with torch.no_grad():
        for x, y in val_loader:
            x, y = x.to(DEVICE), y.to(DEVICE).unsqueeze(1)
            pred = model(x)
            loss = loss_fn(pred, y)
            val_loss += loss.item()

    print(f"epoch {epoch} train loss: {train_loss:.3f} val loss: {val_loss:.3f}")

    scheduler.step(val_loss)

    if val_loss < best_val:
        best_val = val_loss
        counter = 0
        torch.save(model.state_dict(), "../models/best_model.pt")
        print("saved best model")
    else:
        counter += 1
        if counter >= PATIENCE:
            print("early stopping")
            break

epoch 0 train 182.607 val 21.078
saved best model
epoch 1 train 142.969 val 26.333
epoch 2 train 124.226 val 19.869
saved best model
epoch 3 train 110.679 val 18.701
saved best model
epoch 4 train 101.505 val 23.038
epoch 5 train 90.745 val 12.452
saved best model
epoch 6 train 78.890 val 11.116
saved best model
epoch 7 train 70.664 val 10.223
saved best model
epoch 8 train 66.609 val 24.338
epoch 9 train 62.705 val 10.479
epoch 10 train 57.320 val 7.238
saved best model
epoch 11 train 52.666 val 6.961
saved best model
epoch 12 train 51.472 val 6.855
saved best model
epoch 13 train 44.332 val 7.965
epoch 14 train 41.468 val 6.699
saved best model
epoch 15 train 42.013 val 6.132
saved best model
epoch 16 train 41.092 val 5.898
saved best model
epoch 17 train 33.537 val 5.914
epoch 18 train 32.465 val 5.848
saved best model
epoch 19 train 31.903 val 6.156
epoch 20 train 33.431 val 4.792
saved best model
epoch 21 train 31.264 val 5.295
epoch 22 train 30.336 val 5.229
epoch 23 train 27.541

In [9]:
model.load_state_dict(torch.load("../models/best_model.pt"))
model.eval()

correct = 0
total = 0

with torch.no_grad():
    for x, y in test_loader:
        x = x.to(DEVICE)
        y = y.to(DEVICE)

        out = model(x)
        probs = torch.sigmoid(out).squeeze()

        preds = (probs > 0.5).float()
        correct += (preds == y).sum().item()
        total += y.numel()

print("TEST ACCURACY:", correct / total)

TEST ACCURACY: 0.9688888888888889


TEST – final evaluation

What it does:
Loads best saved model and evaluates on test set.

Outputs:

TEST ACCURACY: 0.9688888888888889


Why it exists:
Gives unbiased performance after training.

Validation guides training.
Test tells the truth.

In [ ]:
def predict(wav_path):
    wav = load_wav(wav_path)

    if wav.shape[1] < SAMPLE_RATE:
        wav = F.pad(wav, (0, SAMPLE_RATE - wav.shape[1]))
    else:
        wav = wav[:, :SAMPLE_RATE]

    mel = train_ds.mel(wav)
    mel = train_ds.db(mel)
    mel = mel.unsqueeze(0).to(DEVICE)

    with torch.no_grad():
        out = model(mel)
        prob = torch.sigmoid(out).item()

    return prob